# Stickman Studio · Voice server

Turns your script into a Hindi / Hinglish / English voiceover with **Chatterbox Multilingual** (free, MIT licence) on Colab's free GPU, and can clone **your own voice** from a short recording.

**How to use**
1. Menu **Runtime → Change runtime type → T4 GPU → Save**.
2. Run cell **1** (installs, about 3–5 minutes). The runtime restarts by itself once; that is expected.
3. Run cell **2**. The first time it downloads the model (about 2 GB). It then prints a link like `https://something.trycloudflare.com`.
4. In Stickman Studio → **Script + Voice → AI voice**, paste that link and press **Connect**. Keep this tab open while you make voiceovers.

The link changes every time you start the server. Colab's free GPU stops after a few hours or when the tab is idle for long; then just run cell 2 again.

Only clone your own voice, or a voice whose owner has given you permission. Audio made by Chatterbox carries Resemble AI's inaudible watermark.

In [ ]:
#@title 1 · Install (once per session; restarts the runtime)
import os, subprocess, sys
if os.path.exists('/content/.ss_voice_installed'):
    print('Already installed. Run cell 2.')
else:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'chatterbox-tts==0.1.7', 'fastapi', 'uvicorn'], check=True)
    subprocess.run(['wget', '-q', '-O', '/content/cloudflared',
                    'https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64'], check=True)
    os.chmod('/content/cloudflared', 0o755)
    open('/content/.ss_voice_installed', 'w').write('ok')
    print('Installed. The runtime restarts now (this is expected). Then run cell 2.')
    os.kill(os.getpid(), 9)

In [ ]:
#@title 2 · Start the voice server and get the link
BRANCH = "claude/stickman-video-voiceover-sync-ivss4s" #@param {type:"string"}
import os, re, subprocess, sys, threading, time, urllib.request
import torch
if not torch.cuda.is_available():
    raise SystemExit('No GPU. Menu Runtime → Change runtime type → T4 GPU, then run cell 1 and 2 again.')

urllib.request.urlretrieve(f'https://raw.githubusercontent.com/gopalkrishan680-hash/GK/{BRANCH}/stickman-voice/server.py', '/content/ss_server.py')
sys.path.insert(0, '/content')
import importlib, ss_server
importlib.reload(ss_server)

print('Loading the voice model (first time: downloading about 2 GB)…')
engine = ss_server.start(port=8000)
print('Model ready on', engine.device)

tunnel = subprocess.Popen(['/content/cloudflared', 'tunnel', '--url', 'http://localhost:8000', '--no-autoupdate'],
                          stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
url = None
t0 = time.time()
for line in tunnel.stdout:
    m = re.search(r'https://[a-z0-9-]+\.trycloudflare\.com', line)
    if m:
        url = m.group(0)
        break
    if time.time() - t0 > 90:
        break
threading.Thread(target=lambda: [None for _ in tunnel.stdout], daemon=True).start()  # keep draining the log

if not url:
    print('Could not open the public link. Run this cell again.')
else:
    from IPython.display import HTML, display
    display(HTML(f'''<div style="font:16px sans-serif;padding:14px;border:3px solid #f5c518;border-radius:10px">
      <b>Voice server is running.</b> Paste this link in Stickman Studio → Script + Voice → AI voice:<br>
      <input value="{url}" readonly onclick="this.select()" style="width:100%;font:18px monospace;margin-top:8px;padding:6px">
      <div style="margin-top:6px;color:#666">Keep this tab open while you make voiceovers.</div></div>'''))
    print(url)

### Troubleshooting
- **"No GPU"**: Runtime → Change runtime type → T4 GPU, then run cells 1 and 2 again.
- **Stickman Studio says it cannot reach the server**: the link changes on every start; copy the new one. Also check this tab is still running.
- **A line sounds wrong** (skipped or extra words): in Stickman Studio press **Redo** on that line; it tries again with a new random seed.
- **Hindi sounds English**: write Hindi in Devanagari (हिंदी). English words can stay in English.